# pyPID Demo — PID Controller + FOPDT Simulator

This notebook demonstrates the pyPID enhanced PID controller with:
- FOPDT process simulation
- Mode transitions (Manual → Auto)
- Alarm monitoring
- Interactive trending with Plotly

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pypid import PID, FOPDTSimulator, Mode, AlarmConfig

## 1. Basic PID + FOPDT Step Response

In [ ]:
# Create a PID controller and process simulator
pid = PID(
    Kp=1.5,
    Ki=0.3,
    Kd=0.2,
    setpoint=50.0,
    output_limits=(0, 100),
    sample_time=None,  # update every call
)

sim = FOPDTSimulator(
    K=2.0,     # process gain
    tau=10.0,  # time constant (seconds)
    theta=2.0, # dead time (seconds)
    y0=25.0,   # initial PV
)

# Run simulation
dt = 0.1  # 100ms scan rate
data = []
pv = 25.0

for i in range(1000):
    t = i * dt
    output = pid(pv, dt=dt)
    pv = sim.update(output, dt=dt)
    p, integ, d = pid.components
    data.append({
        'time': t,
        'pv': pv,
        'sp': pid.setpoint,
        'output': output,
        'P': p,
        'I': integ,
        'D': d,
    })

df = pd.DataFrame(data)
print(f"Final PV: {pv:.2f}  |  Setpoint: {pid.setpoint}  |  Steady-state output: {output:.2f}")

In [ ]:
# Plot PV and Output
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Process Variable', 'Controller Output'),
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=df['time'], y=df['pv'], name='PV', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=df['time'], y=df['sp'], name='SP', line=dict(color='red', dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=df['time'], y=df['output'], name='Output', line=dict(color='green')), row=2, col=1)

fig.update_layout(height=500, title_text='PID Step Response with FOPDT Process')
fig.update_xaxes(title_text='Time (s)', row=2, col=1)
fig.update_yaxes(title_text='Temperature (°F)', row=1, col=1)
fig.update_yaxes(title_text='Output (%)', row=2, col=1)
fig.show()

## 2. PID Components Breakdown

In [ ]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df['time'], y=df['P'], name='P term'))
fig2.add_trace(go.Scatter(x=df['time'], y=df['I'], name='I term'))
fig2.add_trace(go.Scatter(x=df['time'], y=df['D'], name='D term'))
fig2.add_trace(go.Scatter(x=df['time'], y=df['output'], name='Total Output', line=dict(dash='dash')))

fig2.update_layout(title='PID Component Contributions', xaxis_title='Time (s)', yaxis_title='Value')
fig2.show()

## 3. Mode Transitions — Manual to Auto with Bumpless Transfer

In [ ]:
# Demonstrate bumpless transfer from Manual to Auto
pid2 = PID(
    Kp=1.5, Ki=0.3, Kd=0.1,
    setpoint=50.0,
    output_limits=(0, 100),
    sample_time=None,
    starting_output=30.0,
)
sim2 = FOPDTSimulator(K=2.0, tau=10.0, theta=2.0, y0=25.0)

dt = 0.1
data2 = []
pv = 25.0

# Start in MANUAL at 30% output
pid2.mode = Mode.MANUAL
pid2.output = 30.0

for i in range(1500):
    t = i * dt
    
    # Switch to AUTO at t=30s
    if i == 300:
        pid2.mode = Mode.AUTO
    
    # Setpoint change at t=80s
    if i == 800:
        pid2.setpoint = 65.0
    
    output = pid2(pv, dt=dt)
    pv = sim2.update(output, dt=dt)
    data2.append({
        'time': t,
        'pv': pv,
        'sp': pid2.setpoint,
        'output': output,
        'mode': pid2.mode.value,
    })

df2 = pd.DataFrame(data2)

fig3 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=('PV and SP', 'Output + Mode'),
                     vertical_spacing=0.08)

fig3.add_trace(go.Scatter(x=df2['time'], y=df2['pv'], name='PV'), row=1, col=1)
fig3.add_trace(go.Scatter(x=df2['time'], y=df2['sp'], name='SP', line=dict(dash='dash')), row=1, col=1)
fig3.add_trace(go.Scatter(x=df2['time'], y=df2['output'], name='Output'), row=2, col=1)

# Add mode transition markers
fig3.add_vline(x=30.0, line_dash='dot', annotation_text='Manual→Auto', row=1, col=1)
fig3.add_vline(x=80.0, line_dash='dot', annotation_text='SP Change', row=1, col=1)

fig3.update_layout(height=500, title_text='Bumpless Transfer: Manual → Auto → SP Change')
fig3.update_xaxes(title_text='Time (s)', row=2, col=1)
fig3.show()

## 4. Alarm Monitoring

In [ ]:
# PID with alarms configured
alarm_cfg = AlarmConfig(
    hsp=55.0,
    hhsp=60.0,
    hhhsp=65.0,
    lsp=35.0,
    llsp=30.0,
    lllsp=25.0,
    yeldev_sp=3.0,
    orgdev_sp=8.0,
)

pid3 = PID(
    Kp=1.5, Ki=0.3, Kd=0.1,
    setpoint=50.0,
    output_limits=(0, 100),
    sample_time=None,
    alarm_config=alarm_cfg,
)
sim3 = FOPDTSimulator(K=2.0, tau=10.0, theta=2.0, y0=25.0)

dt = 0.1
data3 = []
pv = 25.0

for i in range(1000):
    t = i * dt
    output = pid3(pv, dt=dt)
    pv = sim3.update(output, dt=dt)
    alarms = pid3.alarms
    data3.append({
        'time': t,
        'pv': pv,
        'sp': pid3.setpoint,
        'H': alarms.h,
        'HH': alarms.hh,
        'L': alarms.l,
        'LL': alarms.ll,
        'yel_dev': alarms.yel_dev,
        'org_dev': alarms.org_dev,
    })

df3 = pd.DataFrame(data3)

fig4 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=('PV with Alarm Limits', 'Alarm States'),
                     vertical_spacing=0.08)

fig4.add_trace(go.Scatter(x=df3['time'], y=df3['pv'], name='PV'), row=1, col=1)
fig4.add_trace(go.Scatter(x=df3['time'], y=df3['sp'], name='SP', line=dict(dash='dash')), row=1, col=1)
fig4.add_hline(y=55.0, line_dash='dot', line_color='orange', annotation_text='H', row=1, col=1)
fig4.add_hline(y=35.0, line_dash='dot', line_color='cyan', annotation_text='L', row=1, col=1)

fig4.add_trace(go.Scatter(x=df3['time'], y=df3['yel_dev'].astype(int), name='Yellow Dev', line=dict(color='gold')), row=2, col=1)
fig4.add_trace(go.Scatter(x=df3['time'], y=df3['org_dev'].astype(int), name='Orange Dev', line=dict(color='orange')), row=2, col=1)
fig4.add_trace(go.Scatter(x=df3['time'], y=df3['H'].astype(int), name='High', line=dict(color='red')), row=2, col=1)
fig4.add_trace(go.Scatter(x=df3['time'], y=df3['L'].astype(int), name='Low', line=dict(color='blue')), row=2, col=1)

fig4.update_layout(height=500, title_text='PID with Alarm Monitoring')
fig4.update_xaxes(title_text='Time (s)', row=2, col=1)
fig4.update_yaxes(title_text='Active (1/0)', row=2, col=1)
fig4.show()

## 5. Tuning Exploration

Try different Kp/Ki/Kd values and see the effect immediately:

In [ ]:
def run_pid_sim(Kp, Ki, Kd, K_proc=2.0, tau_proc=10.0, theta_proc=2.0, sp=50.0, t_final=100.0, dt=0.1):
    """Run a PID simulation and return a DataFrame of results."""
    pid = PID(Kp=Kp, Ki=Ki, Kd=Kd, setpoint=sp, output_limits=(0, 100), sample_time=None)
    sim = FOPDTSimulator(K=K_proc, tau=tau_proc, theta=theta_proc, y0=sp/2)
    
    data = []
    pv = sp / 2
    n_steps = int(t_final / dt)
    
    for i in range(n_steps):
        output = pid(pv, dt=dt)
        pv = sim.update(output, dt=dt)
        data.append({'time': i*dt, 'pv': pv, 'sp': sp, 'output': output})
    
    return pd.DataFrame(data)

# Compare different tunings
tunings = [
    (1.0, 0.1, 0.0, 'Conservative'),
    (2.0, 0.5, 0.2, 'Moderate'),
    (4.0, 1.0, 0.5, 'Aggressive'),
]

fig5 = go.Figure()
for Kp, Ki, Kd, label in tunings:
    df_t = run_pid_sim(Kp, Ki, Kd)
    fig5.add_trace(go.Scatter(x=df_t['time'], y=df_t['pv'], name=f'{label} (Kp={Kp}, Ki={Ki}, Kd={Kd})'))

fig5.add_hline(y=50.0, line_dash='dash', line_color='gray', annotation_text='SP')
fig5.update_layout(title='Tuning Comparison', xaxis_title='Time (s)', yaxis_title='PV')
fig5.show()